In [2]:
import pandas as pd
import numpy as np

In [3]:
movies=pd.read_csv("../../datasets\movies.csv")
tags=pd.read_csv("../../datasets/tags.csv")

<>:1: SyntaxWarning: invalid escape sequence '\m'
<>:1: SyntaxWarning: invalid escape sequence '\m'
C:\Users\DELL\AppData\Local\Temp\ipykernel_15100\3565466379.py:1: SyntaxWarning: invalid escape sequence '\m'
  movies=pd.read_csv("../../datasets\movies.csv")


In [4]:
movies_tags=movies.merge(tags,on="movieId",how="left")
movies_tags.head(10)

,movieId,title,genres,userId,tag,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,336.0,pixar,1.139046e+09
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,474.0,pixar,1.137207e+09
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,567.0,fun,1.525286e+09
3,2,Jumanji (1995),Adventure|Children|Fantasy,62.0,fantasy,1.528844e+09
4,2,Jumanji (1995),Adventure|Children|Fantasy,62.0,magic board game,1.528844e+09
5,2,Jumanji (1995),Adventure|Children|Fantasy,62.0,Robin Williams,1.528844e+09
6,2,Jumanji (1995),Adventure|Children|Fantasy,474.0,game,1.137376e+09
7,3,Grumpier Old Men (1995),Comedy|Romance,289.0,moldy,1.143425e+09
8,3,Grumpier Old Men (1995),Comedy|Romance,289.0,old,1.143425e+09
9,4,Waiting to Exhale (1995),Comedy|Drama|Romance,NaN,NaN,NaN


In [5]:
movies_tags["tag"].isnull().sum()

np.int64(8170)

In [6]:
movies_tags["tag"]=movies_tags["tag"].fillna(" ")

In [7]:
tags_grouped=(movies_tags.groupby("movieId")["tag"].apply(lambda x:" ".join(x)).reset_index())
tags_grouped.head()

,movieId,tag
0,1,pixar pixar fun
1,2,fantasy magic board game Robin Williams game
2,3,moldy old
3,4,
4,5,pregnancy remake


In [8]:
movie_features=movies.merge(tags_grouped,on="movieId",how="left")
movie_features["tag"]=movie_features["tag"].fillna(" ")

In [10]:
def create_features(row):
    genres=row["genres"]
    tags=row["genres"]
    return f"{genres}{tags}"

In [11]:
movie_features["features"]=movie_features.apply(create_features,axis=1)
movie_features.head(10)

,movieId,title,genres,tag,features
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun,Adventure|Animation|Children|Comedy|FantasyAdv...
1,2,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game Robin Williams game,Adventure|Children|FantasyAdventure|Children|F...
2,3,Grumpier Old Men (1995),Comedy|Romance,moldy old,Comedy|RomanceComedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,,Comedy|Drama|RomanceComedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake,ComedyComedy
5,6,Heat (1995),Action|Crime|Thriller,,Action|Crime|ThrillerAction|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance,remake,Comedy|RomanceComedy|Romance
7,8,Tom and Huck (1995),Adventure|Children,,Adventure|ChildrenAdventure|Children
8,9,Sudden Death (1995),Action,,ActionAction
9,10,GoldenEye (1995),Action|Adventure|Thriller,,Action|Adventure|ThrillerAction|Adventure|Thri...


In [13]:
def preprocess_text(text):
    text=text.lower()
    text=text.replace("|"," ")
    return text

In [15]:
movie_features["features"]=movie_features["features"].apply(preprocess_text)
movie_features.head(10)

,movieId,title,genres,tag,features
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun,adventure animation children comedy fantasyadv...
1,2,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game Robin Williams game,adventure children fantasyadventure children f...
2,3,Grumpier Old Men (1995),Comedy|Romance,moldy old,comedy romancecomedy romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,,comedy drama romancecomedy drama romance
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake,comedycomedy
5,6,Heat (1995),Action|Crime|Thriller,,action crime thrilleraction crime thriller
6,7,Sabrina (1995),Comedy|Romance,remake,comedy romancecomedy romance
7,8,Tom and Huck (1995),Adventure|Children,,adventure childrenadventure children
8,9,Sudden Death (1995),Action,,actionaction
9,10,GoldenEye (1995),Action|Adventure|Thriller,,action adventure thrilleraction adventure thri...


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
def build_content_recommender(movie_features):

    tfidf = TfidfVectorizer(stop_words="english")

    tfidf_matrix = tfidf.fit_transform(
        movie_features["features"]
    )

    similarity_matrix = cosine_similarity(
        tfidf_matrix
    )

    movie_indices = pd.Series(
        movie_features.index,
        index=movie_features["title"]
    ).drop_duplicates()

    return similarity_matrix, movie_indices

In [18]:
similarity_matrix, movie_indices = (
    build_content_recommender(movie_features)
)

In [19]:
def recommend(movie_name,
              movie_indices,
              similarity_matrix,
              data,
              top_n=5):

    idx = movie_indices[movie_name]

    similarity_scores = list(
        enumerate(similarity_matrix[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    movie_indices_list = [
        i[0] for i in similarity_scores
    ]

    return data["title"].iloc[
        movie_indices_list
    ]

In [20]:
recommend(
    "Toy Story (1995)",
    movie_indices,
    similarity_matrix,
    movie_features
)

1706                                       Antz (1998)
2355                                Toy Story 2 (1999)
2809    Adventures of Rocky and Bullwinkle, The (2000)
3000                  Emperor's New Groove, The (2000)
3568                             Monsters, Inc. (2001)
Name: title, dtype: str

In [21]:
movie_features.to_csv(
    "../../datasets/movie_features.csv",
    index=False
)